Unpacking data from catalog unzipped:

In [ ]:
import os
import zipfile
import xarray as xr
import pandas as pd
from pathlib import Path

base_dir = Path(".")
zip_folder = base_dir / "data_raw"
output_folder = base_dir / "data_unzipped"

output_folder.mkdir(exist_ok=True)


for year in range(2000, 2021):
    filename = f"nazare_{year}.nc"
    zip_path = zip_folder / filename
    output_dir = output_folder / f"year_{year}"

    if zip_path.exists():
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(output_dir)
            print(f"Extracted: {filename}")


all_years_data = []

print("\n--- Starting Data Processing ---")

for year_folder in output_folder.iterdir():
    if year_folder.is_dir():
        print(f"Processing folder: {year_folder.name}")
        
        nc_files = list(year_folder.glob("*.nc"))
        
        if not nc_files:
            continue
            
        try:
            ds_year = xr.merge([xr.open_dataset(f) for f in nc_files])
            df_year = ds_year.to_dataframe().reset_index()
            all_years_data.append(df_year)
            
        except Exception as e:
            print(f"Error in folder {year_folder.name}: {e}")

if all_years_data:
    final_df = pd.concat(all_years_data, ignore_index=True)
    final_df = final_df.sort_values(by=["valid_time", "latitude", "longitude"])
    
    print("\n--- Success! All years merged ---")
    print(final_df.info()) 
else:
    print("No data found to process.")